## **Topic classification: AG News, 4 classes (Dataset fancyzhx/ag_news)**

This notebook implements text classification on the **AG News** dataset (4 classes: World, Sports, Business, Sci/Tech) using **BERT (`bert-base-uncased`)**. It covers exploratory data analysis, data preprocessing, a baseline model with Logistic Regression, full fine-tuning using the Hugging Face Trainer, and finally, uploading the trained model to the Hugging Face Hub.

### **Install libraries and check GPU**

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -U transformers datasets evaluate accelerate scikit-learn huggingface_hub

In [ ]:
import torch
import transformers
import datasets
import evaluate
import sklearn
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

### **Load the AG News dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")

print(dataset)
print("\nTrain:", len(dataset["train"]))
print("Test:", len(dataset["test"]))
print("\nFeatures:")
print(dataset["train"].features)

### **Split data (Train, Validation, and Test)**

In [ ]:
from datasets import DatasetDict

SEED = 42

split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="label"
)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": dataset["test"]
})

print(dataset)
print("\nTrain:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

In [ ]:
label_names = dataset["train"].features["label"].names

id2label = {
    i: label for i, label in enumerate(label_names)
}

label2id = {
    label: i for i, label in enumerate(label_names)
}

print("Labels:", label_names)
print("id2label:", id2label)
print("label2id:", label2id)

### **Tokenize the text**

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)

### **Baseline model: Frozen BERT + Logistic Regression**

In [ ]:
import torch
import numpy as np
import time

from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert = AutoModel.from_pretrained(MODEL_NAME)
bert.to(device)

# Freeze all BERT parameters
for param in bert.parameters():
    param.requires_grad = False

bert.eval()

trainable_params = sum(
    p.numel() for p in bert.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in bert.parameters()
)

print("Model:", MODEL_NAME)
print("Total BERT parameters:", total_params)
print("Trainable BERT parameters:", trainable_params)
print("Device:", device)

In [ ]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# Remove the original text column
feature_dataset = tokenized_dataset.remove_columns(["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

train_loader = DataLoader(
    feature_dataset["train"],
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    feature_dataset["validation"],
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    feature_dataset["test"],
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

In [ ]:
def extract_features(model, dataloader, device):
    features = []
    labels = []

    model.eval()

    with torch.no_grad():
        for batch in dataloader:
            labels.append(batch["labels"].numpy())

            batch = {
                key: value.to(device)
                for key, value in batch.items()
                if key in ["input_ids", "attention_mask", "token_type_ids"]
            }

            outputs = model(**batch)

            # [CLS] representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

            features.append(
                cls_embeddings.cpu().numpy()
            )

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels

In [ ]:
start_time = time.time()

X_train, y_train = extract_features(
    bert,
    train_loader,
    device
)

train_feature_time = time.time() - start_time

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print(f"Feature extraction time: {train_feature_time:.2f} seconds")

In [ ]:
start_time = time.time()

X_val, y_val = extract_features(
    bert,
    validation_loader,
    device
)

validation_feature_time = time.time() - start_time

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print(f"Feature extraction time: {validation_feature_time:.2f} seconds")

In [ ]:
start_time = time.time()

X_test, y_test = extract_features(
    bert,
    test_loader,
    device
)

test_feature_time = time.time() - start_time

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print(f"Feature extraction time: {test_feature_time:.2f} seconds")

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs"
)

start_time = time.time()

classifier.fit(X_train, y_train)

classifier_training_time = time.time() - start_time

print(f"Classifier training time: {classifier_training_time:.2f} seconds")
print("Logistic Regression iterations:", classifier.n_iter_)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_predictions = classifier.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

val_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro"
)

print("Validation Accuracy:", round(val_accuracy, 4))
print("Validation Macro F1:", round(val_macro_f1, 4))

In [ ]:
test_predictions = classifier.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro"
)

print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_macro_f1, 4))

In [ ]:
feature_based_results = {
    "task": "Topic Classification",
    "method": "Feature-based + Logistic Regression",
    "train_examples": len(X_train),
    "validation_examples": len(X_val),
    "test_examples": len(X_test),
    "bert_trainable_parameters": 0,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "feature_extraction_seconds": (
        train_feature_time +
        validation_feature_time +
        test_feature_time
    ),
    "classifier_training_seconds": classifier_training_time
}

feature_based_results

In [ ]:
import os

FINAL_MODEL_DIR = "/kaggle/working/models/topic_classification_full_finetuned"

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print("Model directory:")
print(FINAL_MODEL_DIR)

### **Set up full fine-tuning**

In [ ]:
from transformers import AutoModelForSequenceClassification

NUM_LABELS = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Model:", MODEL_NAME)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Device:", device)

In [ ]:
from torch.optim import AdamW

bert_parameters = []
head_parameters = []

for name, parameter in model.named_parameters():
    if name.startswith("bert."):
        bert_parameters.append(parameter)
    else:
        head_parameters.append(parameter)

optimizer = AdamW(
    [
        {
            "params": bert_parameters,
            "lr": 2e-5
        },
        {
            "params": head_parameters,
            "lr": 1e-3
        }
    ],
    weight_decay=0.01
)

print("BERT learning rate: 2e-5")
print("Head learning rate: 1e-3")

In [ ]:
from transformers import TrainingArguments

EPOCHS = 2
BATCH_SIZE = 16
GRADIENT_ACCUMULATION = 2

training_args = TrainingArguments(
    output_dir="./results/topic_full_finetuning",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    weight_decay=0.01,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none"
)

print("Training arguments configured.")
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }


from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    optimizers=(optimizer, None)
)

print("Trainer ready.")

### **Train and evaluate the model**

In [ ]:
import time

start_time = time.time()

train_output = trainer.train()

full_finetuning_time = time.time() - start_time

print(f"Full fine-tuning time: {full_finetuning_time:.2f} seconds")

In [ ]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"]
)

print("Test results:")
print(test_results)

print(f"\nTest Loss: {test_results['eval_loss']:.4f}")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test Macro F1: {test_results['eval_macro_f1']:.4f}")

In [ ]:
import os

model_dir = "/kaggle/working/models/topic_classification_full_finetuned"

os.makedirs(model_dir, exist_ok=True)

# Save the model and tokenizer
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

print("Model saved to:")
print(model_dir)

print("\nSaved files:")
for file in os.listdir(model_dir):
    print(file)

In [ ]:
import os

print("Directory exists:", os.path.isdir(model_dir))
print("Files:", os.listdir(model_dir))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_dir = "/kaggle/working/models/topic_classification_full_finetuned"

# Load the saved model and tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

loaded_model = loaded_model.to("cuda")
loaded_model.eval()

print("Model loaded successfully.")
print("Labels:", loaded_model.config.id2label)

### **Test the model with custom texts**

In [ ]:
texts = [
    # Sports
    "The Canadian women's volleyball team won an exciting match after dominating the final set with strong serves and excellent defense.",

    # Business
    "Stock market investors pushed technology shares higher today as several major companies reported stronger-than-expected quarterly earnings.",

    # Sci/Tech
    "Researchers developed a new artificial intelligence system that analyzes genomic data to help identify genetic patterns associated with disease.",

    # World
    "A major international summit in Ottawa brought together leaders from several countries to discuss Canada's diplomatic relations and cooperation on global issues."
]


# Tokenize the texts
inputs = loaded_tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Move inputs to GPU
inputs = {key: value.to("cuda") for key, value in inputs.items()}

# Make predictions
with torch.no_grad():
    outputs = loaded_model(**inputs)

predictions = torch.argmax(outputs.logits, dim=1)

# Display predictions
for text, prediction in zip(texts, predictions):
    label = loaded_model.config.id2label[prediction.item()]
    print(f"Text: {text}")
    print(f"Predicted topic: {label}")
    print()

### **Check results and confusion matrix**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Get predictions from the fine-tuned model
test_output = trainer.predict(tokenized_dataset["test"])

test_predictions = np.argmax(test_output.predictions, axis=1)
test_labels = test_output.label_ids

# Classification report
print("Classification Report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=label_names,
        digits=4
    )
)

# Confusion matrix
cm = confusion_matrix(test_labels, test_predictions)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="RdPu",
    xticklabels=label_names,
    yticklabels=label_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - BERT Full Fine-Tuning")
plt.tight_layout()
plt.show()

In [ ]:
print("Training history:")

for log in trainer.state.log_history:
    print(log)

In [ ]:
import pandas as pd

results_table = pd.DataFrame([
    {
        "Method": "Feature-based + Logistic Regression",
        "Trainable BERT Parameters": 0,
        "Test Accuracy": 0.9019736842105263,
        "Test Macro F1": 0.9019274036825774,
        "Time (s)": 786.8690655231476 + 105.94409894943237
    },
    {
        "Method": "Full Fine-tuning",
        "Trainable BERT Parameters": 109485316,
        "Test Accuracy": 0.9457894736842105,
        "Test Macro F1": 0.9458511956340154,
        "Time (s)": 2777.6105
    }
])

results_table["Test Accuracy"] = (
    results_table["Test Accuracy"] * 100
).round(2)

results_table["Test Macro F1"] = (
    results_table["Test Macro F1"] * 100
).round(2)

results_table["Time (min)"] = (
    results_table["Time (s)"] / 60
).round(2)

print(results_table.to_string(index=False))

### **Save the model and upload to Hugging Face**

In [ ]:
!pip install -U --no-cache-dir huggingface_hub

In [ ]:
import shutil
import os

model_dir = "/kaggle/working/models/topic_classification_full_finetuned"

shutil.make_archive(
    "/kaggle/working/topic_classification_full_finetuned",
    "zip",
    model_dir
)

print("Backup created:")
print("/kaggle/working/topic_classification_full_finetuned.zip")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os

model_dir = "/kaggle/working/models/topic_classification_full_finetuned"

print("Directory exists:", os.path.isdir(model_dir))

if os.path.isdir(model_dir):
    print("Files:", os.listdir(model_dir))

Directory exists: True
Files: ['config.json', 'training_args.bin', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "karencardiel/classification-bert"

api.upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload fine-tuned BERT topic classification model"
)

print("Model uploaded successfully!")

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

Model uploaded successfully!


In [ ]:
!pip install -U "huggingface-hub>=1.5.0,<2.0"

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.5 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.0.0
    Uninstalling huggingface_hub-2.0.0:
      Successfully uninstalled huggingface_hub-2.0.0


### **Final test: Example Usage**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "karencardiel/topic-classification-bert"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

# Example text
text = "Researchers developed a new artificial intelligence system for analyzing genomic data."

# Tokenize the input
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted class
prediction = torch.argmax(outputs.logits, dim=1).item()

# Convert class ID to label
label = model.config.id2label[prediction]

print("Predicted topic:", label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicted topic: Sci/Tech
